# 第三部分：RAII 与资源生命周期

## 实验 5：异常安全与栈展开

本实验不系统讲解 C++ 异常语法，只研究一个与资源管理直接相关的问题：函数通过 `throw` 非正常退出时，RAII 对象是否仍会释放资源？

答案来自栈展开（stack unwinding）：异常向外寻找匹配的 `catch` 时，会依次销毁沿途已经构造完成的局部对象。所有文件输出都位于 `outputs/05/`。

异常路径的核心规则：

```text
throw
  ↓
离开当前作用域并销毁局部对象
  ↓
逐层退出调用栈并销毁各层局部对象
  ↓
进入匹配的 catch
```

这与普通 `return` 使用同一套对象生命周期规则，但异常可能一次跨越多个函数。

In [ ]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <cstdio>
#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <string>

### 1. 准备可观察的文件资源

`ExceptionFile` 在构造和析构时输出标签。析构函数显式声明为 `noexcept`，表示清理过程不会向外传播异常。

In [ ]:
// 本步骤：通过代码演示“准备可观察的文件资源”并观察结果。
// 创建输出目录，排除路径不存在对异常实验的干扰。
std::filesystem::create_directories("outputs/05");

// 定义可观察的 RAII owner，追踪异常传播期间的清理。
class ExceptionFile
{
public:
    ExceptionFile(
        const std::string &label,
        const std::string &path,
        const char *mode)
        // 构造时保存诊断信息并获取文件句柄。
        : label_(label),
          path_(path),
          file_(std::fopen(path_.c_str(), mode))
    {
        // 文件打开失败就抛出，确保完成构造的对象始终有效。
        if (file_ == nullptr)
        {
            throw std::runtime_error(
                std::string("cannot open file: ") + path_);
        }

        std::cout << label_ << " acquire" << std::endl;
    }

    // 栈展开期间析构不能再抛异常，因此明确保持 noexcept。
    ~ExceptionFile() noexcept
    {
        std::cout << label_ << " release" << std::endl;
        std::fclose(file_);
    }

    // 禁止复制，维持 FILE* 的唯一所有权。
    ExceptionFile(const ExceptionFile &) = delete;
    ExceptionFile &operator=(const ExceptionFile &) = delete;

    // 只借出句柄，不改变 owner，也不转移释放责任。
    std::FILE *get() const noexcept
    {
        return file_;
    }

private:
    std::string label_;
    std::string path_;
    std::FILE *file_;
};

### 2. 异常离开单个函数

`fail_after_open()` 在文件构造完成后抛出异常。函数没有机会执行后续普通语句，但栈展开仍会析构局部文件对象。

In [ ]:
// 本步骤：通过代码演示“异常离开单个函数”并观察结果。
// 获取资源并完成一次写入，然后从函数内部抛出异常。
void fail_after_open()
{
    ExceptionFile file(
        "single frame",
        "outputs/05/output.txt",
        "w");

    std::fputs("written before exception", file.get());
    std::cout << "before exception" << std::endl;

    // 非正常退出触发栈展开，局部 file 会在传播前析构。
    throw std::runtime_error("something went wrong");
}

// 调用失败函数，并在资源释放之后捕获异常。
try
{
    fail_after_open();
}
catch (const std::exception &error)
{
    std::cout << "caught: " << error.what() << std::endl;
}

预期顺序：

```text
single frame acquire
before exception
single frame release
caught: something went wrong
```

`release` 发生在进入 `catch` 之前，证明清理属于异常传播过程的一部分。

### 3. 异常跨越多层调用栈

每个函数都拥有一个局部资源，最深层函数抛出异常。异常向外传播时，会从最内层开始逐层销毁对象。

In [ ]:
// 本步骤：通过代码演示“异常跨越多层调用栈”并观察结果。
// 最深层先获取资源，再抛出触发栈展开的异常。
void deepest_layer()
{
    ExceptionFile file(
        "deepest",
        "outputs/05/deepest.txt",
        "w");
    throw std::runtime_error("failure in deepest layer");
}

// 中间层持有自己的资源，并把调用继续传向深层。
void middle_layer()
{
    ExceptionFile file(
        "middle",
        "outputs/05/middle.txt",
        "w");
    deepest_layer();
}

// 外层最早获取资源，因此会在栈展开中最后释放。
void outer_layer()
{
    ExceptionFile file(
        "outer",
        "outputs/05/outer.txt",
        "w");
    middle_layer();
}

// 从最外层启动调用，在所有局部资源清理后捕获异常。
try
{
    outer_layer();
}
catch (const std::exception &error)
{
    std::cout << "caught: " << error.what() << std::endl;
}

资源先按调用顺序获取，再按相反顺序释放：

```text
outer acquire
middle acquire
deepest acquire
deepest release
middle release
outer release
caught: failure in deepest layer
```

调用方不需要知道每一层函数具体拥有何种资源。每层局部类型只负责自己的清理。

### 4. 构造函数中途失败

如果一个对象有多个 RAII 成员，而后面的成员构造失败，前面已经构造完成的成员仍会自动析构。包含它们的外层对象没有构造完成，因此外层析构函数不会执行。

In [ ]:
// 本步骤：通过代码演示“构造函数中途失败”并观察结果。
// 用探针成员记录成功构造和自动回滚。
class ConstructionProbe
{
public:
    explicit ConstructionProbe(const std::string &label)
        : label_(label)
    {
        std::cout << label_ << " constructed" << std::endl;
    }

    ~ConstructionProbe() noexcept
    {
        std::cout << label_ << " destroyed" << std::endl;
    }

private:
    std::string label_;
};

// 第二个成员在构造时主动失败，模拟部分构造场景。
class FailingMember
{
public:
    FailingMember()
    {
        std::cout << "failing member starts" << std::endl;
        throw std::runtime_error("member construction failed");
    }
};

// 外层对象依次构造 first_ 和 second_，但自身不会完成构造。
class PartialOwner
{
public:
    // first_ 成功后构造 second_；后者抛出时 first_ 自动析构。
    PartialOwner()
        : first_("first member"),
          second_()
    {
        std::cout << "owner constructed" << std::endl;
    }

    // 外层未完成构造时此析构不会运行，日志用于验证规则。
    ~PartialOwner() noexcept
    {
        std::cout << "owner destroyed" << std::endl;
    }

private:
    ConstructionProbe first_;
    FailingMember second_;
};

// 尝试构造外层 owner，并观察失败后的成员回滚与捕获顺序。
try
{
    PartialOwner owner;
}
catch (const std::exception &error)
{
    std::cout << "caught: " << error.what() << std::endl;
}

预期输出不会出现 `owner constructed` 或 `owner destroyed`，因为 `PartialOwner` 没有完成构造；但会出现 `first member destroyed`，因为第一个成员已经构造成功。

这条规则让复杂资源类可以安全地在成员初始化过程中失败：只要每个成员自身是 RAII 类型，已成功获取的资源会自动回滚。

### 5. 析构函数通常不能抛出异常

析构函数承担兜底清理，通常应保持 `noexcept`。如果异常正在栈展开，而某个析构函数又抛出第二个异常，程序会调用 `std::terminate()`，无法由外层普通 `catch` 恢复。

```cpp
~File() noexcept
{
    // best-effort cleanup，不向外抛异常
    std::fclose(file_);
}
```

如果释放操作可能失败且调用方必须获知，可以另外提供显式 `close()` 或 `flush()` 返回错误；析构函数仍负责不抛异常的最终清理。

### 6. 异常安全不只是“没有泄漏”

常见的异常安全保证分为三个层次：

| 保证 | 抛出异常后的状态 |
| --- | --- |
| 基本保证 | 没有资源泄漏，对象仍可析构，但业务状态可能已经改变 |
| 强保证 | 操作要么成功，要么状态保持调用前不变 |
| 不抛保证 | 操作承诺不会抛异常，常用于析构和交换等基础操作 |

RAII 直接帮助实现基本保证中的资源安全，但不会自动回滚已经写入的文件内容或已经提交的数据库状态。强保证通常还需要临时对象、事务或先计算后提交等设计。

### 7. 捕获异常时使用常量引用

本实验统一使用：

```cpp
catch (const std::exception &error)
```

引用避免复制，`const` 表示只观察异常，通过基类引用还能保留动态多态行为。按值捕获基类可能发生对象切片。

### 8. 异常不能直接穿过 C ABI

Native SDK 内部可以使用异常和 RAII，但 C++ 异常不应越过 `extern "C"` 边界传播。边界函数需要捕获所有异常并转换为稳定的错误码或错误对象：

```cpp
extern "C" int sdk_run() noexcept
{
    try
    {
        Runtime runtime;
        runtime.run();
        return SDK_OK;
    }
    catch (const std::exception &error)
    {
        save_last_error(error.what());
        return SDK_ERROR;
    }
    catch (...)
    {
        return SDK_UNKNOWN_ERROR;
    }
}
```

进入 `catch` 之前，函数内部已构造的 RAII 资源已经完成清理。

### 实验结论

异常传播会触发栈展开：从抛出点到匹配的处理器之间，已经构造完成的自动对象按逆序析构。构造中途失败时，已完成构造的成员也会自动清理。

RAII 因而为异常路径提供资源安全基础，但业务状态回滚仍需要额外设计；析构函数应保持不抛异常，跨 C ABI 时则必须在边界捕获并翻译异常。